# **Aprendizagem de Máquina II - 08/10/2026**

## **RBF**

### Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf

### Divisão Treino e Teste

In [ ]:
X, y = make_circles(n_samples=800,
                    noise=0.15,
                    factor=0.4,
                    random_state=42)

X_treino, X_teste, y_treino, y_teste = train_test_split(X,
                                                        y,
                                                        test_size=0.2,
                                                        random_state=42,
                                                        stratify=y
                                                        )

### Scaler

In [ ]:
scaler = StandardScaler()

X_treino_scaled = scaler.fit_transform(X_treino)
X_teste_scaled = scaler.transform(X_teste)

### Plot

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(X_treino_scaled[y_treino == 0, 0],
            X_treino_scaled[y_treino == 0, 1],
            color='blue',
            label='Classe 0 (externa)',
            alpha=0.6,
            edgecolors='black',
            )

plt.scatter(X_treino_scaled[y_treino == 0, 0],
            X_treino_scaled[y_treino == 1, 1],
            color='red',
            label='Classe 1 (interna)',
            alpha=0.6,
            edgecolors='black',
            )
plt.title('Dataset: Make Circles | Circulos Concêntricos')
plt.xlabel('Atributo 1'); plt.ylabel('Atributo 2')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

### Criação de RBF

In [ ]:
class RBFlayer(tf.keras.layers.Layer):
    def __init__(self, units, gamma=1.0, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.gamma = gamma

    def build(self, input_shape):
        self.centers = self.add_weight(name='centers',
                                       shape=(self.units, input_shape[-1]),
                                       initializer='uniform',
                                       trainable=True)

    def call(self, inputs):
        diff = tf.expand_dims(inputs, axis=1) - self.centers
        dist_sq = tf.reduce_sum(tf.square(diff), axis=-1)
        return tf.exp(-self.gamma * dist_sq)

### Criação do Modelo

#### Treino

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input

modelo_rbf = Sequential([
    tf.keras.Input(shape=(2,)),
    RBFlayer(units=10, gamma=0.5, name='camada_rbf'),
    Dense(units=1, activation='sigmoid', name='saida')
])

modelo_rbf.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.05),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

modelo_rbf.summary()

historico = modelo_rbf.fit(
    X_treino_scaled,
    y_treino,
    epochs=60,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

#### Teste

In [ ]:
loss_teste, acc_teste = modelo_rbf.evaluate(
    X_teste_scaled,
    y_teste,
    verbose=0)

print(f'Acurácia no conjunto de teste: {acc_teste * 100:.2f}%')